# LoRA Fine-Tuning on RoBERTa
### Based on: *LoRA: Low-Rank Adaptation of Large Language Models* (Hu et al., 2021)

---

## What this notebook covers

Three things that may appear on the exam:
1. **Full fine-tuning loop** — update all model parameters
2. **LoRA fine-tuning loop** — freeze the base model, only train low-rank A and B matrices
3. **LoRA at inference time** — run the model through its forward pass (no weight merging needed)

All evaluated on MRPC, CoLA, and STS-B from the GLUE benchmark.

In [ ]:
# Install dependencies (run once in Colab)
!pip install -q transformers datasets scipy scikit-learn

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, RobertaModel
from datasets import load_dataset
from scipy.stats import pearsonr, matthews_corrcoef
from sklearn.metrics import accuracy_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type != 'cuda':
    print('No GPU detected. In Colab: Runtime → Change runtime type → T4 GPU')

---

## Part 1 — Background: What is LoRA and Why?

**The problem with full fine-tuning:**  
When you fine-tune a model like RoBERTa (125M params) or GPT-3 (175B params) on a new task,  
full fine-tuning updates *every* parameter. That means:
- A 350 GB checkpoint *per task* for GPT-3
- ~1.2 TB of GPU memory during training
- No weight sharing across tasks

**LoRA's key insight:**  
The *change* in weights during adaptation has a low "intrinsic rank" — it lives in a small subspace.  
Instead of learning the full `ΔW` matrix, we learn a low-rank approximation: `ΔW = B · A`.

**The math:**  
For a weight matrix `W₀ ∈ R^(d × k)`, the modified forward pass is:
```
h = W₀x + ΔWx = W₀x + BAx
```
Where:
- `A ∈ R^(r × k)` — initialized with random Gaussian values
- `B ∈ R^(d × r)` — initialized to **zero** so `ΔW = 0` at training start
- `r` is the rank (a small number, like 4 or 8)
- The output is scaled by `α/r` (a constant that stabilizes training across different ranks)

**Parameter savings (example with r=8, d=k=768):**
```
Full ΔW:  768 × 768 = 589,824 params
LoRA ΔW:  (8×768) + (768×8) = 12,288 params   →  48× fewer!
```

**Why B=0?**  
So the model starts identical to the pre-trained model — a stable starting point.

**Where to apply LoRA in a Transformer?**  
The paper applies LoRA to the **query (Wq) and value (Wv)** projection matrices in each attention layer.  
Spreading across both Q and V works better than using a larger rank on just one.

```
Input x
   │
   ├─────────────────────────┐
   │                         │
   ▼                         ▼
W₀ (frozen)              A (r × k)  ← trainable, Gaussian init
   │                         │
   │                         ▼
   │                     B (d × r)  ← trainable, zero init
   │                         │
   ▼                         ▼
 W₀x          +        (α/r) · BAx
   │                         │
   └────────────┬────────────┘
                ▼
            Output h
```

---

## Part 2 — LoRA Implementation

We implement LoRA as a wrapper around `nn.Linear`. The wrapper:
1. **Freezes** the original weight (no gradient, no update)
2. **Adds** two small trainable matrices A and B
3. **Computes** the LoRA forward pass: `W₀x + (α/r) · BAx`

In [ ]:
class LoRALinear(nn.Module):
    """
    Wraps an existing nn.Linear with a LoRA low-rank update.

    Forward pass: h = W₀x + (alpha/r) * B @ A @ x
      - W₀ is frozen (no gradient updates)
      - A and B are the only trainable parameters
      - B is initialized to zero so ΔW = 0 at training start
    """

    def __init__(self, linear: nn.Linear, r: int = 8, alpha: int = 16):
        super().__init__()
        self.linear  = linear
        self.r       = r
        self.scaling = alpha / r  # the α/r factor from the paper

        in_features  = linear.in_features   # k
        out_features = linear.out_features  # d

        # A: (r × k) — Gaussian init, learns which input directions matter
        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.02)
        # B: (d × r) — zero init so ΔW = B@A = 0 at the start of training
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

        # Freeze the original weight — it will not receive any gradient
        self.linear.weight.requires_grad_(False)
        if self.linear.bias is not None:
            self.linear.bias.requires_grad_(False)

    def forward(self, x):
        # Base path through the original frozen weight
        base_out = self.linear(x)

        # LoRA path: x @ A^T gives shape (..., r)
        #            then @ B^T gives shape (..., out_features)
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T

        return base_out + self.scaling * lora_out


def apply_lora_to_roberta(model: nn.Module, r: int = 8, alpha: int = 16):
    """
    Replace the query and value projection linears in every RoBERTa
    attention layer with LoRALinear wrappers.

    The paper applies LoRA to Wq and Wv (not Wk or Wo) and finds
    this gives the best efficiency/performance trade-off.
    """
    for layer in model.roberta.encoder.layer:
        attn = layer.attention.self
        attn.query = LoRALinear(attn.query, r=r, alpha=alpha)
        attn.value = LoRALinear(attn.value, r=r, alpha=alpha)
    return model

---

## Part 3 — Model: RoBERTa with a Classification Head

RoBERTa is a pre-trained transformer. We attach a linear head on top of its `[CLS]` token output:

```
Input tokens:  [CLS]  word1  word2  ...  [SEP]
                 │
         RoBERTa encoder (12 transformer layers)
                 │
         [CLS] hidden state (768-dim)
                 │
             Dropout(0.1)
                 │
         Linear(768 → num_labels)
                 │
             Logits / Prediction
```

- `num_labels=2` for binary classification (MRPC, CoLA)
- `num_labels=1` for regression (STS-B)

In [ ]:
class RobertaClassifier(nn.Module):
    """
    RoBERTa with a linear classification or regression head on the [CLS] token.
    """

    def __init__(self, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.roberta    = RobertaModel.from_pretrained('roberta-base')
        hidden          = self.roberta.config.hidden_size  # 768
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        # Take the [CLS] token representation (position 0)
        cls_hidden = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls_hidden))


def count_parameters(model):
    """Returns (total, trainable) parameter counts."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

---

## Part 4 — Datasets

We use three tasks from the GLUE benchmark:

| Dataset | Input | Label | Metric |
|---------|-------|-------|--------|
| **MRPC** | Two sentences | 0/1 (same meaning?) | Accuracy |
| **CoLA** | One sentence | 0/1 (grammatical?) | Matthews Correlation Coefficient (MCC) |
| **STS-B** | Two sentences | 0.0–5.0 (similarity) | Pearson Correlation |

All inputs are tokenized to max length 128 using RoBERTa's tokenizer.  
STS-B labels are normalized from `[0, 5]` → `[0, 1]` for numerical stability during MSE training.

In [ ]:
MAX_LEN    = 128
BATCH_SIZE = 32

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')


def tokenize_pair(batch, key1, key2):
    """Tokenize a sentence pair into one sequence separated by </s>."""
    return tokenizer(
        batch[key1], batch[key2],
        truncation=True, padding='max_length', max_length=MAX_LEN
    )


def tokenize_single(batch, key):
    """Tokenize a single sentence."""
    return tokenizer(
        batch[key],
        truncation=True, padding='max_length', max_length=MAX_LEN
    )


def get_dataloader(dataset, label_col, batch_size=BATCH_SIZE, shuffle=True):
    """Wrap a HuggingFace dataset split in a PyTorch DataLoader."""
    dataset = dataset.with_format('torch')

    def collate(batch):
        input_ids      = torch.stack([b['input_ids']      for b in batch])
        attention_mask = torch.stack([b['attention_mask'] for b in batch])
        labels         = torch.tensor([b[label_col]       for b in batch])
        return input_ids, attention_mask, labels

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)


print('Loading datasets...')

# MRPC — sentence pairs, binary labels
mrpc_raw = load_dataset('glue', 'mrpc')
mrpc     = mrpc_raw.map(lambda b: tokenize_pair(b, 'sentence1', 'sentence2'), batched=True)
mrpc     = mrpc.remove_columns(['sentence1', 'sentence2', 'idx']).rename_column('label', 'labels')
mrpc_train = get_dataloader(mrpc['train'],      'labels')
mrpc_val   = get_dataloader(mrpc['validation'], 'labels', shuffle=False)
print(f'MRPC   — train: {len(mrpc["train"]):,}  val: {len(mrpc["validation"]):,}')

# CoLA — single sentence, binary label
cola_raw = load_dataset('glue', 'cola')
cola     = cola_raw.map(lambda b: tokenize_single(b, 'sentence'), batched=True)
cola     = cola.remove_columns(['sentence', 'idx']).rename_column('label', 'labels')
cola_train = get_dataloader(cola['train'],      'labels')
cola_val   = get_dataloader(cola['validation'], 'labels', shuffle=False)
print(f'CoLA   — train: {len(cola["train"]):,}  val: {len(cola["validation"]):,}')

# STS-B — sentence pairs, continuous labels normalized to [0, 1]
stsb_raw = load_dataset('glue', 'stsb')
stsb = stsb_raw.map(lambda b: {**b, 'label': [v / 5.0 for v in b['label']]}, batched=True)
stsb = stsb.map(lambda b: tokenize_pair(b, 'sentence1', 'sentence2'), batched=True)
stsb = stsb.remove_columns(['sentence1', 'sentence2', 'idx']).rename_column('label', 'labels')
stsb_train = get_dataloader(stsb['train'],      'labels')
stsb_val   = get_dataloader(stsb['validation'], 'labels', shuffle=False)
print(f'STS-B  — train: {len(stsb["train"]):,}  val: {len(stsb["validation"]):,}')

print('Done!')

---

## Part 5 — Training and Evaluation

### Loss functions
- **Classification** (MRPC, CoLA): **Cross-Entropy Loss** — measures how wrong the class probabilities are
- **Regression** (STS-B): **MSE Loss** — penalizes squared distance from the true score

### Optimizer
**AdamW** (Adam with weight decay) is used for all experiments, matching the paper.  
Learning rates differ by method:
- Full fine-tuning: `lr = 2e-5` — small, to not overwrite pre-trained knowledge
- LoRA: `lr = 2e-4` — larger, since A and B start at zero and need faster convergence

### Evaluation metrics
- **Accuracy** (MRPC): `correct / total`
- **Matthews Correlation Coefficient** (CoLA): handles class imbalance better than accuracy. Range: -1 to +1.
- **Pearson Correlation** (STS-B): measures linear correlation between predicted and true scores. Range: -1 to +1.

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, task):
    """One pass over the training set. Returns average loss."""
    model.train()
    total_loss = 0.0

    for input_ids, attention_mask, labels in loader:
        input_ids      = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels         = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)

        if task == 'regression':
            loss = loss_fn(logits.squeeze(-1), labels.float())
        else:
            loss = loss_fn(logits, labels.long())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, task):
    """
    Evaluate the model. Returns the task-specific metric:
      'accuracy'   → classification accuracy          (MRPC)
      'matthews'   → Matthews Correlation Coefficient  (CoLA)
      'regression' → Pearson correlation               (STS-B)
    """
    model.eval()
    all_preds, all_labels = [], []

    for input_ids, attention_mask, labels in loader:
        input_ids      = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        logits = model(input_ids, attention_mask)

        if task == 'regression':
            preds = logits.squeeze(-1).cpu().numpy()
        else:
            preds = logits.argmax(dim=-1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

    if task == 'accuracy':
        return accuracy_score(all_labels, all_preds)
    elif task == 'matthews':
        return matthews_corrcoef(all_labels, all_preds)
    elif task == 'regression':
        r, _ = pearsonr(all_labels, all_preds)
        return r


def run_training(model, train_loader, val_loader, task, epochs=3, lr=2e-4, label=''):
    """
    Full training loop.
    Only trains parameters where requires_grad=True,
    so it works identically for both full fine-tuning and LoRA.
    Returns the best validation metric seen across all epochs.
    """
    loss_fn   = nn.MSELoss() if task == 'regression' else nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.01
    )

    metric_name = {'accuracy': 'Acc', 'matthews': 'MCC', 'regression': 'Pearson r'}[task]
    best_metric = -float('inf')

    for epoch in range(1, epochs + 1):
        loss   = train_epoch(model, train_loader, optimizer, loss_fn, task)
        metric = evaluate(model, val_loader, task)
        print(f'  {label} Epoch {epoch}/{epochs}  loss={loss:.4f}  val {metric_name}={metric:.4f}')
        best_metric = max(best_metric, metric)

    return best_metric

---

## Part 6 — Full Fine-Tuning

In full fine-tuning, **all** model parameters are updated — the entire 125M params of RoBERTa-base  
plus the new classification head. This is our upper-bound baseline.

**Paper targets (Table 2, RoBERTa-base full fine-tune):**
| MRPC (Acc) | CoLA (MCC) | STS-B (Pearson r) |
|:---:|:---:|:---:|
| ~90.9% | ~63.6% | ~91.2% |

In [ ]:
results = {}  # collect all results for the final comparison table

# ── Full fine-tuning: MRPC ────────────────────────────────────────────────────
print('Full Fine-Tuning — MRPC (Accuracy)')
model_full_mrpc = RobertaClassifier(num_labels=2).to(DEVICE)
total, trainable = count_parameters(model_full_mrpc)
print(f'  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
results['Full_MRPC'] = run_training(
    model_full_mrpc, mrpc_train, mrpc_val,
    task='accuracy', epochs=3, lr=2e-5, label='MRPC-FT'
)
print()

In [ ]:
# ── Full fine-tuning: CoLA ────────────────────────────────────────────────────
print('Full Fine-Tuning — CoLA (Matthews Correlation)')
model_full_cola = RobertaClassifier(num_labels=2).to(DEVICE)
results['Full_CoLA'] = run_training(
    model_full_cola, cola_train, cola_val,
    task='matthews', epochs=3, lr=2e-5, label='CoLA-FT'
)
print()

In [ ]:
# ── Full fine-tuning: STS-B ───────────────────────────────────────────────────
print('Full Fine-Tuning — STS-B (Pearson Correlation)')
print('  (labels normalized to [0,1]; model outputs a single scalar)')
model_full_stsb = RobertaClassifier(num_labels=1).to(DEVICE)
results['Full_STSB'] = run_training(
    model_full_stsb, stsb_train, stsb_val,
    task='regression', epochs=3, lr=2e-5, label='STSB-FT'
)
print()

---

## Part 7 — LoRA Fine-Tuning

Now we repeat the same experiments with LoRA. The only changes are:

1. **Freeze** all of RoBERTa's parameters first (`requires_grad = False`)
2. **Inject** LoRA into Wq and Wv of every attention layer — these A and B matrices are trainable
3. The **classifier head** stays trainable (it's randomly initialized and must learn from scratch)
4. Use a **higher learning rate** (2e-4) since the LoRA matrices start at zero

**Hyperparameters (from paper Appendix D):**
- Rank `r = 8`, scaling `α = 16` → scaling factor = α/r = 2
- 3 epochs, batch size 32, max length 128

**Paper targets (Table 2, RoBERTa-base LoRA):**
| MRPC (Acc) | CoLA (MCC) | STS-B (Pearson r) |
|:---:|:---:|:---:|
| ~89.7% | ~63.4% | ~91.5% |

Being within 1–2 points is acceptable.

In [ ]:
LORA_R     = 8
LORA_ALPHA = 16

# ── LoRA: MRPC ────────────────────────────────────────────────────────────────
print(f'LoRA Fine-Tuning — MRPC  (r={LORA_R}, α={LORA_ALPHA})')
model_lora_mrpc = RobertaClassifier(num_labels=2).to(DEVICE)

# Step 1: freeze all of RoBERTa
for p in model_lora_mrpc.roberta.parameters():
    p.requires_grad_(False)

# Step 2: inject LoRA — adds trainable A and B into each Q and V layer
apply_lora_to_roberta(model_lora_mrpc, r=LORA_R, alpha=LORA_ALPHA)

total, trainable = count_parameters(model_lora_mrpc)
print(f'  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

results['LoRA_MRPC'] = run_training(
    model_lora_mrpc, mrpc_train, mrpc_val,
    task='accuracy', epochs=3, lr=2e-4, label='MRPC-LoRA'
)
print()

In [ ]:
# ── LoRA: CoLA ────────────────────────────────────────────────────────────────
print(f'LoRA Fine-Tuning — CoLA  (r={LORA_R}, α={LORA_ALPHA})')
model_lora_cola = RobertaClassifier(num_labels=2).to(DEVICE)
for p in model_lora_cola.roberta.parameters():
    p.requires_grad_(False)
apply_lora_to_roberta(model_lora_cola, r=LORA_R, alpha=LORA_ALPHA)

results['LoRA_CoLA'] = run_training(
    model_lora_cola, cola_train, cola_val,
    task='matthews', epochs=3, lr=2e-4, label='CoLA-LoRA'
)
print()

In [ ]:
# ── LoRA: STS-B ───────────────────────────────────────────────────────────────
print(f'LoRA Fine-Tuning — STS-B  (r={LORA_R}, α={LORA_ALPHA})')
model_lora_stsb = RobertaClassifier(num_labels=1).to(DEVICE)
for p in model_lora_stsb.roberta.parameters():
    p.requires_grad_(False)
apply_lora_to_roberta(model_lora_stsb, r=LORA_R, alpha=LORA_ALPHA)

results['LoRA_STSB'] = run_training(
    model_lora_stsb, stsb_train, stsb_val,
    task='regression', epochs=3, lr=2e-4, label='STSB-LoRA'
)
print()

---

## Part 8 — Results: Table 2 Comparison

In [ ]:
def pct(key):
    return results[key] * 100

print('=' * 68)
print('  Results vs. Paper Table 2 (RoBERTa-base)')
print('=' * 68)
print(f'  {"":<22}  {"MRPC (Acc)":>12}  {"CoLA (MCC)":>12}  {"STS-B (Pear)":>13}')
print('-' * 68)
print(f'  {"Paper — Full FT":<22}  {"~90.9%":>12}  {"~63.6%":>12}  {"~91.2%":>13}')
print(f'  {"Ours  — Full FT":<22}  {pct("Full_MRPC"):>11.1f}%  {pct("Full_CoLA"):>11.1f}%  {pct("Full_STSB"):>12.1f}%')
print('-' * 68)
print(f'  {"Paper — LoRA r=8":<22}  {"~89.7%":>12}  {"~63.4%":>12}  {"~91.5%":>13}')
print(f'  {"Ours  — LoRA r=8":<22}  {pct("LoRA_MRPC"):>11.1f}%  {pct("LoRA_CoLA"):>11.1f}%  {pct("LoRA_STSB"):>12.1f}%')
print('=' * 68)

# Parameter summary
total_full, train_full = count_parameters(model_full_mrpc)
total_lora, train_lora = count_parameters(model_lora_mrpc)
print()
print('  Trainable parameter comparison:')
print(f'    Full FT : {train_full:>9,}  ({100*train_full/total_full:.1f}% of model)')
print(f'    LoRA    : {train_lora:>9,}  ({100*train_lora/total_lora:.2f}% of model)  →  {train_full//train_lora}× fewer')
print()
print('  Note: results within ~1-2% of paper targets are expected and acceptable.')

---

## Part 9 — LoRA at Inference Time

At inference, **no special handling is needed** — the LoRA layers are already part of the forward pass.  
The `LoRALinear.forward()` method automatically computes `W₀x + (α/r)·BAx` on every call.

Just call the model normally with `model.eval()` and `torch.no_grad()`.

In [ ]:
# ── MRPC inference example ────────────────────────────────────────────────────
model_lora_mrpc.eval()

sentence1 = "The cat sat on the mat."
sentence2 = "A cat was sitting on a mat."

enc = tokenizer(
    sentence1, sentence2,
    return_tensors='pt',
    truncation=True, padding='max_length', max_length=MAX_LEN
)

with torch.no_grad():
    logits = model_lora_mrpc(
        enc['input_ids'].to(DEVICE),
        enc['attention_mask'].to(DEVICE)
    )

probs = torch.softmax(logits, dim=-1).squeeze()
pred  = logits.argmax(dim=-1).item()
label_map = {0: 'Not paraphrase', 1: 'Paraphrase'}

print('LoRA Inference (MRPC):')
print(f'  Sentence 1 : "{sentence1}"')
print(f'  Sentence 2 : "{sentence2}"')
print(f'  Prediction : {label_map[pred]}')
print(f'  Confidence : not-para={probs[0]:.3f}  para={probs[1]:.3f}')
print()
print('Note: the LoRA layers (LoRALinear) are called as-is in the forward pass.')
print('No weight merging is needed for inference.')

In [ ]:
# ── STS-B inference example ───────────────────────────────────────────────────
model_lora_stsb.eval()

sent_a = "Two men are playing guitar."
sent_b = "A man is playing a musical instrument."

enc2 = tokenizer(
    sent_a, sent_b,
    return_tensors='pt',
    truncation=True, padding='max_length', max_length=MAX_LEN
)

with torch.no_grad():
    score = model_lora_stsb(
        enc2['input_ids'].to(DEVICE),
        enc2['attention_mask'].to(DEVICE)
    ).squeeze().item()

# Clamp to [0, 1] and convert back to [0, 5] scale for interpretability
score_clamped    = max(0.0, min(1.0, score))
score_original   = score_clamped * 5.0

print('LoRA Inference (STS-B):')
print(f'  Sentence 1 : "{sent_a}"')
print(f'  Sentence 2 : "{sent_b}"')
print(f'  Raw model output (0-1 scale) : {score:.4f}')
print(f'  Similarity score (0-5 scale) : {score_original:.2f}')

---

## Quick Reference: Common Bugs to Watch For

These are the kinds of things you might be asked to debug on the exam.

### Bug 1 — LoRA matrices not frozen correctly
```python
# ✗ WRONG: freezing after apply_lora means A and B also get frozen
apply_lora_to_roberta(model)
for p in model.roberta.parameters():
    p.requires_grad_(False)   # This freezes lora_A and lora_B too!

# ✓ CORRECT: freeze first, then apply LoRA
for p in model.roberta.parameters():
    p.requires_grad_(False)
apply_lora_to_roberta(model)  # A and B are created with requires_grad=True
```

### Bug 2 — Wrong LoRA forward pass
```python
# ✗ WRONG: missing the base output
def forward(self, x):
    return (x @ self.lora_A.T) @ self.lora_B.T  # forgot W₀x!

# ✗ WRONG: wrong matrix multiply order (A then B, not B then A)
def forward(self, x):
    return self.linear(x) + self.scaling * (x @ self.lora_B.T) @ self.lora_A.T

# ✓ CORRECT: base + scaled LoRA update
def forward(self, x):
    return self.linear(x) + self.scaling * (x @ self.lora_A.T) @ self.lora_B.T
```

### Bug 3 — Wrong loss for STS-B
```python
# ✗ WRONG: using CrossEntropyLoss for regression
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)  # labels are floats, not class indices!

# ✓ CORRECT: use MSELoss and squeeze the logits
loss_fn = nn.MSELoss()
loss = loss_fn(logits.squeeze(-1), labels.float())
```

### Bug 4 — B initialized to Gaussian instead of zero
```python
# ✗ WRONG: ΔW ≠ 0 at training start, unstable initialization
self.lora_B = nn.Parameter(torch.randn(out_features, r) * 0.02)

# ✓ CORRECT: B=0 so ΔW = BA = 0, model starts identical to pre-trained
self.lora_B = nn.Parameter(torch.zeros(out_features, r))
```

### Bug 5 — Optimizer includes frozen parameters
```python
# ✗ WRONG: passing all parameters, wasting compute on frozen weights
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

# ✓ CORRECT: filter to only trainable parameters
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4
)
```